# Assignment 4: Regularization

In [1]:
import urllib.request
import os
import zipfile
import os

def download_data(force=False):
    """Download and extract course data from Zenodo."""

    zip_path = 'data.zip'
    data_dir = './data'

    if not os.path.exists(zip_path) or force:
        print("Downloading course data...")
        urllib.request.urlretrieve(
            'https://zenodo.org/records/18235955/files/data.zip?download=1',
            zip_path
        )
        print("Download complete")

    if not os.path.exists(data_dir) or force:
        print("Extracting data files...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(data_dir)
        print("Data extracted")

    return data_dir


if __name__ == "__main__":
    download_data()


Download complete
Extracting data files...
Data extracted


**Q1.** Please answer the following questions in your own words.

1. What is the intuition of adding a penalty to mean squared error, that grows in the "size" (absolute or squared value) of the model parameters?

The intuition of adding a penalty to mean squared error that grows in the size of the model parameters is to attempt to reduce the efficiency of overly complex models to prevent overfitting. The model attempting to create the lowest possible error could possibly use many complex variables to lower the error as much as possible, but this ends up in a model that is highly sensitive to noise and is not very generalizable. By introducing this penalty, you make the model balance having a model that is a good "fit" and a model that is simple, which may lead to more error in the training set but will ultimately be more generalizable to other data. This way, the model will focus on the key trends and patterns that have a significant impact on the data rather than attempting to focus on every possible bit of noise that might help it "game" the error.

2. How does regularization provide a way of exploring the bias-variance trade-off?

Regularization provides a way to explore the bias-variance trade-off by introducing a penalty parameter that represents just how much error or punishment we are putting onto the model. The balance between bias and variance is directly represented in the balance bewteen minimizing the error and keeping the model simple in this context. For example, an alpha value that is low may allow a model that has a leaning towards maximizing accuracy while assuming high variance, while a regularization model that has a very high alpha value correlating to punishment may favor a model that has less accuracy but lower variance, improving generalizibility. Ultimately, regularization through manipulation of the punishment values and the parameters is a direct exploration of this trade-off.

3. What is the difference between LASSO and Ridge regression? How do the answers typically change for the two problems?

Though LASSO and Ridge regression are both based on the concept of adding a "penalty" to the model for overcomplexity, they have different mechanisms of function. Firstly, the method of calculation is different. LASSO takes absolute values while Ridge will take squared values. This math is different. Secondly, LASSO has the capability to push coefficents to exactly 0 rapidly, isolating variables and creating more sudden changes to improve the model's predicting power. This makes a simpler and more interpretable model quickly while choosing a model specifiction for you, but can make it susceptible to small changes in data as the model can swing in structure fast. On the other hand, Ridge gradually shrinks the effect of coefficents but does not have the same zeroing power that LASSO does, making the changes to the model more gradual as the data changes. This is naturally more stable, producing smoother coefficient estimate values for smaller changes in data. This results in Ridge having better predictive power overall in terms of reliability, while LASSO gives a more interpretable model sooner as it drops variables entirely if it maximizes the result, isolating what is the truly impactful trends.

4. How do we typically scale variables for use in regularized regression? Why?

We typically scale numeric variables with a z-score standardization to better fit with the math of the penalty and to prevent regularization from altering coefficients inappropriately due to scale rather than actual prediction. Since coefficient values are considered in the model here, the scale of the data values matter because data values with vastly different scales and what an incremental increase or decrease means will skew the model's predictive power since fundamentally, the scales are not standardized among different variables. This means that the size of the coefficient may not always accurately reflect the true impact of the feature since the model is trying to balance arbitrary units rather than performance. A change in one unit does not mean the same thing across vastly different units, making numeric variables not always easily comparable. A standardization will put things on a normalized, standard deviation scale, meaning the model can now directly compare coefficients to each other more validly and adjust the strength that way. Ultimately, it is important to ensure that the penalty is being applied fairly across each coefficient to prevent mismatched copmarisons, or for lack of a better term, trying to compare apples and oranges that exist on different scales.

5. How is the penalty $\alpha$ typically selected?

With reference to the lecture notes, the predictive alpha value is crucial as it determines, in simple terms, the penalty and how much it is going to weigh in the model. An alpha value too low will not be able to cause a meaningful change from an ordinary linear regression while an alpha value too high will push coefficents to zero excessively since coefficients will not be able to provide predictive power to justify the cost of the penalty in this situation. A middle area is considered a sweet spot and this is chosen by cross validaiton. Essentially, we pick a grid of alpha values, use a k-fCV for each value (picking the coefficients to minimize MSE(b-hat) + alpha|b-hat| or MSE(b-hat) + alpha(b-hat)^2 and then evaluating MSE(b-hat) on test folds and saving this estimate value), pick a mean or median value of this alpha value  for that MSE(b-hat), and then we compare these values across the grid andchoose an alpha value that gives the best trade-off bewteen bias and variance. Interestingly, this also gives a path of the coefficients as the alpha value changes.

6. When conducting cross validation, do you include the penalty term in evaluating the cross validated MSE? Why or why not?

We do not include the penalty term in evaluating the cross validated MSE. Ultimately, the isolated MSE is the predictive power of the model, and the alpha value and penalties are only used in the training / model building phase to guide coefficient allocation. The penalty is an artificial change we are making on the model building phase to drive change but this will actually "bias" or skew our estimations if we try to include this in the final predictive evaluation. Some models might falsely look better than others in cross validation simply because of the size of the coefficients if this penalty is included, which means we may end up choosing alpha values and models that are actuall worse at raw predictive strength because we are trying to take coefficient size into consideration. This also defeats the purpose of cross validation, which is to compare true predictive power, not account for an artificial penalty value used in model creation.

**Q2.** This is a case study on regularization.

1. Import the `cars_hw.csv` dataset. Create an `Age` variable for each vehicle. Take `Mileage_Run` and `Age`, and (a) use `PolynomialFeatures` to create a third degree expansion, (b) use `StandardScaler` to $z$-score normalize them.


2. Use your features, run linear regression. What is the sign for the interaction between `Mileage_Run` and `Age`?


3. Use `LassoCV` to regularize your linear regression, using 20-fold cross validation. (Hint: I used the grid `alphas = np.logspace(1,3,20)` to find the cost parameter)
4. Plot the cross-validated MSE by $\alpha$.
5. Plot the coefficient paths by $\alpha$.
6. Which features are actually selected? What proportion are set equal to zero?
7. Compare the linear regressions and optimally regularized coefficients. Do any coefficients increase in magnitude from linear regression to LASSO? Do any change sign?

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

cars = pd.read_csv("/content/data/cars_hw.csv")

year = 2026
cars["Age"] = year - cars["Make_Year"]

X = cars[["Mileage_Run", "Age"]]

poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X)
poly_names = poly.get_feature_names_out()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_poly)

print(X_scaled)

[[ 0.12469203 -0.01309242 -0.13767332 ... -0.23076715 -0.19558597
  -0.27042227]
 [-0.87233418  0.34185769 -0.79131562 ... -0.71818608 -0.56489035
   0.04408937]
 [-0.49351672 -0.72299265 -0.60062321 ... -0.64800377 -0.70091085
  -0.71839826]
 ...
 [ 0.46248973  1.05175792  0.19479598 ...  0.32073381  0.73673979
   0.88897673]
 [-0.54294553 -1.07794277 -0.62951357 ... -0.69364241 -0.80062127
  -0.86578933]
 [-0.01621083 -0.36804254 -0.25974731 ... -0.38445401 -0.42380175
  -0.52226369]]


**Q3.** This is a case study on regularization.

1. Import the `heart_failure_clinical_records_dataset.csv` dataset. Use `PolynomialFeatures` to create a third-degree expansion of `age`, `ejection_fraction`, and `serum_creatinine`, and then use `StandardScaler` to $z$-score normalize your results. Use `PolynomialFeatures` with `interaction_only=True` to interact the dummy/categorical variables `anaemia`, `diabetes`, `high_blood_pressure`, and `smoking`. Concatenate these results into your feature/covariate matrix.
2. Use your features, run linear regression. Are there any sign patterns that appear counterintuitive? Why? Can you see how the inclusion of higher-order powers or interactions might resolve the apparent contradiction?
3. Use `LassoCV` to regularize your linear regression, using 20-fold cross validation. (Hint: I used the grid `alphas = np.logspace(-5,5,30)` to find the cost parameter)
4. Plot the cross-validated MSE by $\alpha$.
5. Plot the coefficient paths by $\alpha$.
6. Which features are actually selected? What proportion are set equal to zero? Compare the linear regressions and optimally regularized coefficients. Do any coefficients increase in magnitude from linear regression to LASSO? Do any change sign? Do the sign patterns for the linear_model or the Lasso seem to make more sense? Explain why this might be the case from the perspective of the bias-variance trade-off.

**Q4.** To better understand the math of regularization, we'll solve the regularized linear model problem with a single explanatory variable. So, the model is
$$
\tilde{y}_i = \tilde{b}_0 + \tilde{b}_1 \tilde{x}_i,
$$
where
$$
\tilde{y}_i = y_i - \bar{y} \quad \text{ and } \quad \tilde{x}_i = x_i - \bar{x}.
$$

Recall, we do this mean-normalization of $x$ and $y$, because
$$
\frac{1}{n} \sum_{i=1}^n \tilde{y} = \frac{1}{n} \sum_{i=1}^n y_i - \bar{y} = 0,
$$
and likewise for $x$. This trick makes the calculations easier and the results more easily interpretable.

1. To do ridge regression, add a penalty $+ \alpha (b_1)^2$ to mean squared error. What is the objective function for this problem?
2. Take the derivatives of your objective function with respect to $b_0$ and $b_1$. Set these equations equal to zero. Solve the two equations in two unknowns for $b_1$ and $b_0$.
3. How does increasing $\alpha$ change the slope coefficient?
4. If we instead used the LASSO/L1 penalty, $+\alpha |b_1|$, what challenge do you run into? This is conceptually difficult, but take 5 minutes and try to figure out the solution, and in particular, when is it optimal to set $b_1=0$?